## CS 224: week 2 code demo

Continuing from week 1. Super-relevant stuff for HW1!! 
- Linear algebra in Pytorch
- introducing models / "Modules"
- Downloading and inspecting a pretrained model ("module" object)

**Look in Modules on Canvas to get code after lecture.**

In [1]:
import torch as t

## Linear algebra / tensor manipulation in pytorch

In [2]:
# Construct some matrices / tensors
x = t.tensor([5.])

In [3]:
t.ones((3))
t.zeros((2))

tensor([0., 0.])

In [4]:
x = t.arange(5)

In [5]:
x = t.randn((3,2))

In [6]:
x

tensor([[-0.5438, -1.0083],
        [-0.4734, -1.6088],
        [ 1.0693,  0.6675]])

In [7]:
# Manipulate tensors

In [10]:
x

tensor([[-0.5438, -1.0083],
        [-0.4734, -1.6088],
        [ 1.0693,  0.6675]])

In [9]:
t.cat([x, t.ones((3,1))], dim=1)

tensor([[-0.5438, -1.0083,  1.0000],
        [-0.4734, -1.6088,  1.0000],
        [ 1.0693,  0.6675,  1.0000]])

In [16]:
a = t.ones(2)
b = t.zeros(2)
print(a,b)
t.cat([a,b])
t.stack([a,b], dim=0)

tensor([1., 1.]) tensor([0., 0.])


tensor([[1., 1.],
        [0., 0.]])

In [26]:
# Mathematical operations
print(x)
t.sum(x, dim=0)
t.mean(x, dim=0)

tensor([[-0.5438, -1.0083],
        [-0.4734, -1.6088],
        [ 1.0693,  0.6675]])


tensor([ 0.0174, -0.6499])

In [30]:
x.mean(dim=0)

tensor([ 0.0174, -0.6499])

In [35]:
t.log(x).isfinite()

tensor([[False, False],
        [False, False],
        [ True,  True]])

In [45]:
# broadcasting
print(x.shape, a.shape, b.shape)
c = 5 * t.ones(3)
print(c.unsqueeze(1))
x + c.unsqueeze(1)

torch.Size([3, 2]) torch.Size([2]) torch.Size([2])
tensor([[5.],
        [5.],
        [5.]])


tensor([[4.4562, 3.9917],
        [4.5266, 3.3912],
        [6.0693, 5.6675]])

In [50]:
# Matrix/tensor multiplication - so many possibilities!\

print(x.shape, a.shape)
t.matmul(x,a)
x@a
t.einsum('ij,j->i', x, a)  # fancy!
t.tensordot(x, a, dims=[[1], [0]])  # super fancy

torch.Size([3, 2]) torch.Size([2])


tensor([-1.5521, -2.0822,  1.7368])

In [53]:
# Other linear algebra operations
M = x.T @ x
print(M.shape, M)
t.linalg.inv(M)

torch.Size([2, 2]) tensor([[1.6633, 2.0237],
        [2.0237, 4.0505]])


tensor([[ 1.5333, -0.7661],
        [-0.7661,  0.6296]])

### Introducing "Modules"
Besides tensors, the next basic building block is a flexible and powerful way to specify models.

In [54]:
class MyNet(t.nn.Module):
    def __init__(self, input_size):
        super().__init__()  # Important! 
        self.bias = t.nn.Parameter(t.tensor(5.))
    
    def forward(self, x):
        return x + self.bias

In [55]:
model = MyNet(2)

In [60]:
with t.no_grad():
    print(model(10))

tensor(15.)


### Working with a pretrained model

In [61]:
from transformers import AutoTokenizer, AutoModel

model_name="bert-base-uncased"  # Many possibilities on huggingface.com

# Load pre-trained model tokenizer (vocabulary)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pre-trained model
model = AutoModel.from_pretrained(model_name)

sentence = "Hello, this is an example sentence!"

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [88]:
# Look at the tokenizer
sentence = "Hello, this right here is an example sentence!"
input_ids = tokenizer(sentence, return_tensors='pt').input_ids
# tokenizer.convert_ids_to_tokens([2019, 2742])
print(input_ids)

tensor([[ 101, 7592, 1010, 2023, 2157, 2182, 2003, 2019, 2742, 6251,  999,  102]])


In [89]:
# Inspect the model
with t.no_grad():
    out = model(input_ids).last_hidden_state
print(out.shape)
print(out[0].mean(dim=0).shape)

torch.Size([1, 12, 768])
torch.Size([768])


In [106]:
# Now lets build something to get the embeddings
def embed_sentence(model, tokenizer, sentence):
    """Function to embed a sentence as a vector using a pre-trained model."""
    inputs = tokenizer(sentence, return_tensors="pt")  # Turn text into one-hot vectors

    # Forward pass, get hidden states
    with t.no_grad():
        outputs = model(**inputs)

    # For BERT, last hidden state is the embedding of each item in the sequence
    return outputs.last_hidden_state[0].mean(dim=0)  # use mean embedding, for hw
    # return outputs.last_hidden_state[0, 0]  # use CLS embedding, another good choice

In [107]:
embedding = embed_sentence(model, tokenizer, sentence)

In [108]:
embedding.shape

torch.Size([768])